# Part 3 — Graph Initialization & Management

*Last updated:* 2026-11-12

This notebook shows how to **build and cache** an IDTrack graph snapshot for:
- `homo_sapiens` (human)
- `mus_musculus` (mouse)
- `sus_scrofa` (pig)
- `macaca_mulatta` (rhesus macaque) and `macaca_fascicularis` (cynomolgus macaque)

A graph build is the most expensive step. The good news:
- you usually do it **once per organism + snapshot boundary + external YAML configuration**
- the snapshot is **multi-assembly** (human has overlapping GRCh38/GRCh37; mouse, pig and the macaques are clean-handoff by release, and
  the ids of their earlier assemblies are in the graph too)
- then you load the graph file from your local repository in a few seconds, without network

**Learning objectives**
- Build (or load) a graph snapshot for each organism.
- Verify that the snapshot exists on disk.
- Learn practical graph-management habits (reload vs rebuild, cache hygiene).

> **Prerequisite:** every supported organism ships a default external YAML; run `02_prepare_new_external_yaml.ipynb` first only if
> you want your own selection of external databases.


## 3.0 — What you should expect (time / disk)

Measured with this version (snapshot release 115, the shipped external YAML files, a graph built from nothing over HTTPS on 2 cores of
a Linux server; the build time is mostly download time and depends on your network):

| | pig | rhesus | cynomolgus | mouse | human |
| --- | ---: | ---: | ---: | ---: | ---: |
| build from nothing | 15 min | 14 min | 6 min | 36 min | 1 h 55 min |
| peak memory of the build | 1.1 GB | 1.0 GB | 0.6 GB | 2.9 GB | 4.5 GB |
| tables on disk | 0.65 GB | 0.66 GB | 0.27 GB | 2.1 GB | 5.3 GB |
| graph file | 85 MB | 69 MB | 47 MB | 228 MB | 345 MB |
| loading the graph file | 1.3 s | 1.2 s | 0.9 s | 2.8 s | 4.3 s |
| memory after loading | 0.38 GB | 0.34 GB | 0.26 GB | 0.76 GB | 1.14 GB |

Plan for a first build like you would plan for downloading a reference genome + annotation; after that, a graph loads in seconds
(the first load of a file on a machine took up to 7 s).

In [1]:
# Load notebook utilities (collapsible output magic for tutorials). `_notebook_utils.py` sits next to the notebooks in
# IDTrack's repository; a notebook downloaded on its own runs without it and shows the logs as they come.
try:
    get_ipython().run_line_magic('load_ext', '_notebook_utils')
except ModuleNotFoundError:
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def collapse(line, cell):
        get_ipython().run_cell(cell)

In [2]:
# 1) Setup
from __future__ import annotations

import os
from pathlib import Path

import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

print('Local repository:', LOCAL_REPOSITORY)


Local repository: idtrack_cache


## 3.0.1 — Sanity check: which external YAML files will be used?

Every supported organism ships a packaged default (`idtrack/default_config/<organism>_externals_modified.yml`); a file of the same
name in your local repository (created in Part 2) takes precedence.

In [3]:
# External YAML presence: local repository first (created in Part 2), else the packaged default
DEFAULT_CONFIG = Path(idtrack.__file__).parent / 'default_config'


def _yaml_status(organism):
    local = LOCAL_REPOSITORY / f'{organism}_externals_modified.yml'
    shipped = DEFAULT_CONFIG / local.name
    if local.exists():
        return True, 'OK', local.name
    if shipped.exists():
        return True, 'OK (packaged default)', local.name
    return False, 'MISSING (create it in Part 2)', local.name


HAS_HUMAN_YAML, *_human = _yaml_status('homo_sapiens')
HAS_MOUSE_YAML, *_mouse = _yaml_status('mus_musculus')
HAS_PIG_YAML, *_pig = _yaml_status('sus_scrofa')

for status, name in (_human, _mouse, _pig):
    print(status.ljust(55), name)

OK                                                      homo_sapiens_externals_modified.yml
OK                                                      mus_musculus_externals_modified.yml
OK                                                      sus_scrofa_externals_modified.yml


If a file is missing:
- go back to `02_prepare_new_external_yaml.ipynb`
- generate the template and create the `_modified.yml` file


## 3.1–3.3 — Build graph snapshots (one per organism)

The canonical pattern is:

1. resolve organism name
2. pick (and pin) the snapshot release
3. `api.build_graph(...)`: loads the graph file if it is in the local repository, builds and saves it otherwise
4. inspect + reuse

The primary assembly of a graph is the organism's preferred assembly that has the snapshot release (human GRCh38, mouse GRCm39, pig
Sscrofa11.1, rhesus Mmul_10, cynomolgus 6.0 at release 115); it is part of the graph file's name.

We do this for each organism below. If you only need one organism, run only that section. The tutorials pin the snapshot release
to 115: `resolve_organism` returns Ensembl's newest release, which changes every few months, and a graph is built per snapshot.


### 3.1 — Human graph initialization (multi-assembly)

The human graph has GRCh38 as its primary assembly (assembly key `38`): conversions use, at each release, the assembly Ensembl maintained
at that release (NCBI36 `36` to release 54, GRCh37 `37` to 75, GRCh38 from 76), and the GRCh37 ids of later releases are in the graph as
ids to convert from.

This is what enables atlas-building workflows where different datasets were annotated with different genome builds, but you want one unified
identifier space.


In [4]:
organism, latest_release = api.resolve_organism('human')
SNAPSHOT_RELEASE = 115  # pinned: the same graph and the same answers in every run (latest_release is Ensembl's newest)
organism, SNAPSHOT_RELEASE


2026-11-12 15:15:15 INFO:idtrack._verify_organism: Ensembl Rest API query to get the organism names and associated releases.


('homo_sapiens', 115)

In [5]:
%%collapse Click to show build logs
# Included for tutorial purposes only.

# Build (or load) the graph snapshot
api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE)


In [6]:
# Quick inspection
g = api.track.graph
print('Organism:', g.graph.get('organism'))
print('Snapshot release:', g.graph.get('ensembl_release'))
print('Main assembly:', g.graph.get('genome_assembly'))
print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))
print('Nodes:', g.number_of_nodes())
print('Edges:', g.number_of_edges())

aed = sorted(getattr(g, 'available_external_databases', []))
print('External DBs enabled (count):', len(aed))
print('External DBs (first 20):', aed[:20])

# What the graph file records about itself (checked when it is loaded)
manifest = g.graph['manifest']
print('Graph format version:', manifest['format_version'], '| built by idtrack', manifest['package_version'])
print('History corrections made while building:', manifest['history']['corrections'])


Organism: homo_sapiens
Snapshot release: 115
Main assembly: 38
Assemblies in this graph: [36, 37, 38]
Nodes: 3682041


Edges: 8630251
External DBs enabled (count): 27
External DBs (first 20): ['CCDS', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'Havana translation', 'NCBI gene', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'UniProtKB Gene Name', 'Uniprot/SPTREMBL', 'Uniprot/SWISSPROT']
Graph format version: 2 | built by idtrack 0.0.5
History corrections made while building: [{'edges': 7, 'events': 7, 'recorded': [114, 114], 'used': [114, 115]}]


In [7]:
# Where is the graph file stored?
sorted(LOCAL_REPOSITORY.glob('graph_homo_sapiens*.npz'))[-5:]


[PosixPath('idtrack_cache/graph_homo_sapiens_assembly38_min48_max115_narrow.npz')]

### 3.2 — Mouse graph initialization (clean handoff)

Mouse is a clean-handoff species (one maintained assembly per release: NCBIM37 → GRCm38 → GRCm39). Older assemblies mainly matter for
legacy datasets and archive releases; you typically do not have overlapping assemblies within the same release.


In [8]:
organism, latest_release = api.resolve_organism('mus musculus')
SNAPSHOT_RELEASE = 115
organism, SNAPSHOT_RELEASE


('mus_musculus', 115)

In [9]:
%%collapse Click to show build logs
# Included for tutorial purposes only.

# Build (or load) the mouse graph snapshot
if HAS_MOUSE_YAML:
    api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE)
else:
    print('Skipping mouse build: mus_musculus_externals_modified.yml is missing (run Part 2 first).')


In [10]:
if HAS_MOUSE_YAML:
    g = api.track.graph
    print('Organism:', g.graph.get('organism'))
    print('Snapshot release:', g.graph.get('ensembl_release'))
    print('Main assembly:', g.graph.get('genome_assembly'))
    print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))
    print('Nodes:', g.number_of_nodes())
    print('Edges:', g.number_of_edges())

    aed = sorted(getattr(g, 'available_external_databases', []))
    print('External DBs enabled (count):', len(aed))
    print('External DBs (first 20):', aed[:20])
else:
    print('Mouse graph not built (missing YAML).')


Organism: mus_musculus
Snapshot release: 115
Main assembly: 39
Assemblies in this graph: [37, 38, 39]
Nodes: 2476278


Edges: 5367306
External DBs enabled (count): 27
External DBs (first 20): ['CCDS', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'Havana gene', 'Havana transcript', 'Havana translation', 'MGI Symbol', 'NCBI gene', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'UniProtKB Gene Name', 'Uniprot/SPTREMBL', 'Uniprot/SWISSPROT']


In [11]:
sorted(LOCAL_REPOSITORY.glob('graph_mus_musculus*.npz'))[-5:]


[PosixPath('idtrack_cache/graph_mus_musculus_assembly39_min48_max115_narrow.npz')]

### 3.3 — Pig graph initialization (clean handoff)

Pig is a clean-handoff species (one maintained assembly per release: Sscrofa9 → Sscrofa10.2 → Sscrofa11.1). Older assemblies mainly
matter for legacy datasets and archive releases; you typically do not have overlapping assemblies within the same release.


In [12]:
organism, latest_release = api.resolve_organism('sus scrofa')
SNAPSHOT_RELEASE = 115
organism, SNAPSHOT_RELEASE


('sus_scrofa', 115)

In [13]:
%%collapse Click to show build logs
# Included for tutorial purposes only.

# Build (or load) the pig graph snapshot
if HAS_PIG_YAML:
    api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE)
else:
    print('Skipping pig build: sus_scrofa_externals_modified.yml is missing (run Part 2 first).')


In [14]:
if HAS_PIG_YAML:
    g = api.track.graph
    print('Organism:', g.graph.get('organism'))
    print('Snapshot release:', g.graph.get('ensembl_release'))
    print('Main assembly:', g.graph.get('genome_assembly'))
    print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))
    print('Nodes:', g.number_of_nodes())
    print('Edges:', g.number_of_edges())

    aed = sorted(getattr(g, 'available_external_databases', []))
    print('External DBs enabled (count):', len(aed))
    print('External DBs (first 20):', aed[:20])
else:
    print('Pig graph not built (missing YAML).')


Organism: sus_scrofa
Snapshot release: 115
Main assembly: 111
Assemblies in this graph: [9, 102, 111]
Nodes: 1073214


Edges: 1757774
External DBs enabled (count): 24
External DBs (first 20): ['Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'NCBI gene', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'UniProtKB Gene Name', 'Uniprot/SPTREMBL', 'Uniprot/SWISSPROT', 'VGNC Symbol', 'synonym_id::EntrezGene']


In [15]:
sorted(LOCAL_REPOSITORY.glob('graph_sus_scrofa*.npz'))[-5:]


[PosixPath('idtrack_cache/graph_sus_scrofa_assembly111_min56_max115_narrow.npz')]

### 3.4 — Rhesus and cynomolgus macaque graph initialization (clean handoff)

Both macaques are clean-handoff species:
- rhesus (`macaca_mulatta`): MMUL_1 (key `1`, releases 48–85) → Mmul_8.0.1 (`801`, 86–97) → Mmul_10 (`10`, 98+)
- cynomolgus (`macaca_fascicularis`): Macaca_fascicularis_5.0 (`5`, 91–102) → 6.0 (`6`, 103+)

The snapshot graph links the assemblies through Ensembl's own history (e.g. rhesus 85 → 86 and 97 → 98), so identifiers from an
older genome build convert to the current one. Gene symbols are VGNC symbols on the current assemblies and HGNC symbols on the older
ones; both are enabled in the shipped YAML files. Rhesus and cynomolgus are two organisms and two graphs.

The full builds read every release since 48 (rhesus) or 91 (cynomolgus); they work over HTTPS alone (`IDTRACK_ENSEMBL_ACCESS=https`).

In [16]:
HAS_MMUL_YAML, *_mmul = _yaml_status('macaca_mulatta')
HAS_MFAS_YAML, *_mfas = _yaml_status('macaca_fascicularis')
for status, name in (_mmul, _mfas):
    print(status.ljust(55), name)

OK                                                      macaca_mulatta_externals_modified.yml
OK                                                      macaca_fascicularis_externals_modified.yml


In [17]:
%%collapse Click to show build logs
# Included for tutorial purposes only.

# Build (or load) the rhesus macaque graph snapshot
organism, latest_release = api.resolve_organism('macaca_mulatta')
if HAS_MMUL_YAML:
    api.build_graph(organism_name=organism, snapshot_release=115)
    print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))
else:
    print('Skipping rhesus build: macaca_mulatta_externals_modified.yml is missing (run Part 2 first).')

In [18]:
%%collapse Click to show build logs
# Included for tutorial purposes only.

# Build (or load) the cynomolgus macaque graph snapshot
organism, latest_release = api.resolve_organism('macaca_fascicularis')
if HAS_MFAS_YAML:
    api.build_graph(organism_name=organism, snapshot_release=115)
    print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))
else:
    print('Skipping cynomolgus build: macaca_fascicularis_externals_modified.yml is missing (run Part 2 first).')

## 3.5 — Graph management (all species)

### Reloading vs rebuilding

- `api.build_graph(...)` is safe to call repeatedly.
  - If the graph file is in the local repository, IDTrack **loads** it: from the file alone, with no table, no database manager and
    no network.
  - If it does not exist yet, IDTrack **builds** it (slow, first time only) and saves it.
- One `idtrack.API` holds one graph: `build_graph` for another organism replaces the graph of the object. Keep one `API` per organism
  if you need several at once.

### What a graph file is

- One `.npz` file per graph, `graph_<organism>_assembly<primary assembly>_min<first release>_max<snapshot>_narrow.npz`: graphs of
  different organisms, snapshots and primary assemblies sit side by side in one local repository.
- The file records what it was built from: the selection of external databases that takes effect (from your external YAML), the organism
  table, the graph format and the package version, the tables, and the corrections made to Ensembl's id history
  (`api.track.graph.graph['manifest']`). Loading a file under the expected name that was built from something else (another YAML
  selection, another organism table, an older graph format) raises `idtrack.GraphMismatchError`, which names the differences: IDTrack
  never answers from another graph and never starts a build of hours on its own. To build it again, move the file away and call
  `build_graph`.
- Graph files of IDTrack 0.0.5 (`graph_*.pickle`) and its table cache (`<organism>_assembly-<key>.h5`) are not read; build the graphs
  again with this version.

### Cache hygiene

Your local repository accumulates:
- downloaded tables (`idtrack_tables/<organism>_assembly-<key>/`, one file per table; a table records what it depends on, so changing the
  YAML or the window never serves a stale table and nothing has to be deleted by hand)
- graph files (`graph_*.npz`)
- your external YAML files

> **Tip:** Treat your local repository as *project infrastructure*. Keep it stable so you get the benefits of caching.

### Switching organisms / profiles

A graph is specific to:
- organism
- snapshot boundary (max release) and window
- the external databases its YAML selects (`<organism>_externals_modified.yml` of the local repository, or the shipped default)

To compare two selections of external databases, use one local repository per selection (the YAML file name is fixed per repository; see
Part 7).

### Performance tips

- Keep your external YAML allowlist small to reduce ambiguity and the size of the downloads.

> **Warning:** Deleting the tables of a local repository means downloading them again for the next build.

In [19]:
# Helper: list what IDTrack has cached in your local repository.
# Safe: this does NOT delete anything.

from pathlib import Path

cache = LOCAL_REPOSITORY

print('Local repository:', cache)

patterns = [
    'graph_*.npz',
    'idtrack_tables/*',
    '*_externals_modified.yml',
    '*_externals_template.yml',
]

for pat in patterns:
    hits = sorted(cache.glob(pat))
    print()
    print(f'{pat} ({len(hits)}):')
    for p in hits[:10]:
        print('  ', p.name)
    if len(hits) > 10:
        print('  ...')


Local repository: idtrack_cache

graph_*.npz (5):
   graph_homo_sapiens_assembly38_min48_max115_narrow.npz
   graph_macaca_fascicularis_assembly6_min91_max115_narrow.npz
   graph_macaca_mulatta_assembly10_min48_max115_narrow.npz
   graph_mus_musculus_assembly39_min48_max115_narrow.npz
   graph_sus_scrofa_assembly111_min56_max115_narrow.npz

idtrack_tables/* (14):
   homo_sapiens_assembly-36
   homo_sapiens_assembly-37
   homo_sapiens_assembly-38
   macaca_fascicularis_assembly-5
   macaca_fascicularis_assembly-6
   macaca_mulatta_assembly-1
   macaca_mulatta_assembly-10
   macaca_mulatta_assembly-801
   mus_musculus_assembly-37
   mus_musculus_assembly-38
  ...

*_externals_modified.yml (5):
   homo_sapiens_externals_modified.yml
   macaca_fascicularis_externals_modified.yml
   macaca_mulatta_externals_modified.yml
   mus_musculus_externals_modified.yml
   sus_scrofa_externals_modified.yml

*_externals_template.yml (0):
